## Import Libraries

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 3, Finished, Available, Finished, False)

### Bronze Total count

In [2]:
display(spark.table('Bronze.API_Raw_Data').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 4, Finished, Available, Finished, False)

931927

### Silver Count Before Streaming

In [3]:
display(spark.table('Silver.API_silver_Data').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 5, Finished, Available, Finished, False)

793186

### Create a Streaming Dataframe

In [4]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_bronze=spark.readStream.format('delta').load("abfss://Kiran@onelake.dfs.fabric.microsoft.com/RT_Project.Lakehouse/Tables/Bronze/API_Raw_Data")

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 6, Finished, Available, Finished, False)

### Remove the Null Responses from API

In [5]:
df_valid=df_bronze.filter(trim(col('results'))!='[]')

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 7, Finished, Available, Finished, False)

### Normalize the Response to Flatten the Json object

In [6]:
df_normalized=df_valid.withColumn('results',
                when(
                    trim(col('results')).startswith("["),
                    trim(col('results'))).
                otherwise(
                    concat(lit('['),trim(col('results')),
                    lit(']')
                    )
                )
            )

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 8, Finished, Available, Finished, False)

### Extract all the fields from Json Object

In [7]:
df_silver=df_normalized.select(
        get_json_object(col('results'),"$[0].gender").alias('Gender'),
        get_json_object(col('results'),"$[0].name.title").alias('Title'),
        get_json_object(col('results'),"$[0].name.first").alias('First'),
        get_json_object(col('results'),"$[0].name.last").alias('Last'),
        concat(get_json_object(col('results'),"$[0].location.street.number"),lit(', '),get_json_object(col('results'),"$[0].location.street.name")).alias('Street_Info'),
        get_json_object(col('results'),"$[0].location.city").alias('City'),
        get_json_object(col('results'),"$[0].location.state").alias('State'),
        get_json_object(col('results'),"$[0].location.country").alias('Country'),
        get_json_object(col('results'),"$[0].location.postcode").alias('Postcode'),
        get_json_object(col('results'),"$[0].location.coordinates.latitude").alias('Latitude'),
        get_json_object(col('results'),"$[0].location.coordinates.longitude").alias('Longitude'),
        get_json_object(col('results'),"$[0].location.timezone.offset").alias('TimeZone_Offset'),
        get_json_object(col('results'),"$[0].location.timezone.description").alias('TimeZone_Description'),
        get_json_object(col('results'),"$[0].email").alias('Email'),
        get_json_object(col('results'),"$[0].login.uuid").alias('userid'),
        get_json_object(col('results'),"$[0].login.username").alias('User_Name'),
        get_json_object(col('results'),"$[0].login.password").alias('Password'),
        get_json_object(col('results'),"$[0].login.salt").alias('Salt'),
        get_json_object(col('results'),"$[0].login.md5").alias('MD5'),
        get_json_object(col('results'),"$[0].login.sha1").alias('SHA1'),
        get_json_object(col('results'),"$[0].login.sha256").alias('SHA256'),
        get_json_object(col('results'),"$[0].dob.date").alias('Birth_Date'),
        get_json_object(col('results'),"$[0].dob.age").alias('Age'),
        get_json_object(col('results'),"$[0].registered.date").alias('Registered_Date'),
        get_json_object(col('results'),"$[0].registered.age").alias('Registered_Age'),
        get_json_object(col('results'),"$[0].phone").alias('Phone_Number'),
        get_json_object(col('results'),"$[0].cell").alias('Cell_Number'),
        get_json_object(col('results'),"$[0].id.name").alias('ID_Name'),
        get_json_object(col('results'),"$[0].id.value").alias('ID_Value'),
        get_json_object(col('results'),"$[0].picture.large").alias('Picture_Large'),
        get_json_object(col('results'),"$[0].picture.medium").alias('Picture_Medium'),
        get_json_object(col('results'),"$[0].picture.thumbnail").alias('Picture_thumbnail'),
        get_json_object(col('results'),"$[0].nat").alias('NAT'),
        col('injestion_timestamp').alias('Injestion_Timestamp'),
        current_timestamp().alias("Processing_Timestamp")

        )

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 9, Finished, Available, Finished, False)

### Write the streaming data to Silver Layer

In [8]:
query=(df_silver.writeStream
                .format('delta')
                .outputMode("append")
                .option('checkpointLocation',
                        'Files/checkpoint/api_silver')
                .toTable("Silver.API_Silver_Data")
    )

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 10, Finished, Available, Finished, False)

In [9]:
spark.streams.active

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 11, Finished, Available, Finished, False)

### Silver Count After Streaming

In [15]:
display(spark.table('Silver.API_silver_Data').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 46, Finished, Available, Finished, False)

932682